# 01 · Segmentation Visualizer — Baseline vs. Optimized (test set)

Visual inspection of the lesion masks predicted on the **held-out test set** in Phase 5.
Each row shows the dermoscopy image and the overlay of the **ground truth** and the **prediction**
for the Baseline (Phase 1) and the Optimized (Phase 4) model of the same size. Overlays are drawn on a
lightened greyscale copy of the image so the colours stay readable on dark lesions.

| Overlay | Meaning | Secondary cue (colour-vision safe) |
|---|---|---|
| green  | ground truth only (missed lesion, FN) | solid outline = ground-truth border |
| red    | prediction only (false positive, FP) | dashed outline = predicted border |
| yellow | ground truth ∩ prediction (TP) | — |

**Inputs** (written by `run_pipeline.sh --phases 5`):
`phase5_test/per_image/<variant>_<model>_fp32.csv` (per-image DSC/JSI) and
`phase5_test/masks/<variant>_<model>/<image>.png` (predicted masks, FP32).
Ground truth is rasterised from the YOLO labels with the same function used for the metrics
(`yolo26_seg/segmentation_metrics.py`), so what you see is exactly what was scored.
No GPU is needed.

In [ ]:
# ---- Configuration ----------------------------------------------------------
PIPELINE_NAME = "pipeline_final_v1"   # sub-directory of logs/ (or set $PIPELINE_DIR)
MODEL = "small"                       # nano | small | medium | large | xlarge
PRECISION = "fp32"                    # masks are saved for FP32 only
N_SAMPLES = 6                         # random test images to show
N_EXTREMES = 4                        # rows in the "largest gain / regression / hardest" panels
SEED = 0                              # reproducible sampling

# Overlay colours (requested: GT green, prediction red; overlap yellow) + outline styles
GT_COLOR, PRED_COLOR, OVERLAP_COLOR = "#00c853", "#ff1744", "#ffd600"
ALPHA = 0.40


import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Locate the repository and the pipeline outputs ------------------------
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def find_pipeline_dir(name: str) -> Path:
    """$PIPELINE_DIR, else the Docker mount, else <repo>/logs/<name>."""
    candidates = [os.environ.get("PIPELINE_DIR"), f"/workspace/logs/{name}", REPO_ROOT / "logs" / name]
    for c in candidates:
        if c and Path(c).is_dir():
            return Path(c)
    raise FileNotFoundError(f"Pipeline outputs not found; tried {candidates}. Set PIPELINE_DIR.")


def local_path(p) -> Path:
    """Map /workspace/... paths recorded inside Docker to this checkout when run on the host."""
    p = Path(p)
    if not p.exists() and str(p).startswith("/workspace/"):
        alt = REPO_ROOT / p.relative_to("/workspace")
        if alt.exists():
            return alt
    return p


PIPELINE_DIR = find_pipeline_dir(PIPELINE_NAME)
FIG_DIR = PIPELINE_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

# ---- Figure style (publication, light) -------------------------------------
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#ffffff"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "semibold", "axes.titlelocation": "left",
    "axes.labelcolor": INK_2, "axes.edgecolor": INK_2, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "legend.frameon": False, "legend.fontsize": 8, "lines.linewidth": 2, "lines.markersize": 7,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def save(fig, name: str) -> None:
    """Save a figure as vector PDF (for LaTeX) and 300-dpi PNG under <pipeline>/figures/."""
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=300)
    print(f"saved {FIG_DIR / name}.pdf|png")


print("pipeline :", PIPELINE_DIR)
print("figures  :", FIG_DIR)

In [ ]:
import cv2
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

sys.path.insert(0, str(REPO_ROOT / "yolo26_seg"))
from segmentation_metrics import label_path_for, rasterize_yolo_label  # noqa: E402

VARIANTS = ("baseline", "optimized")
PER_IMAGE = PIPELINE_DIR / "phase5_test" / "per_image"

available = sorted({p.stem.split("_")[1] for p in PER_IMAGE.glob(f"*_{PRECISION}.csv")})
if MODEL not in available:
    print(f"[info] no Phase 5 results for MODEL={MODEL!r}; available: {available}")
    MODEL = available[0]
    print(f"[info] using MODEL={MODEL!r}")


def load_variant(variant: str) -> pd.DataFrame:
    df = pd.read_csv(PER_IMAGE / f"{variant}_{MODEL}_{PRECISION}.csv")
    return df[["image", "dsc", "jsi", "n_pred", "empty_pred"]].rename(
        columns=lambda c: c if c == "image" else f"{c}_{variant}")


per_image = load_variant("baseline").merge(load_variant("optimized"), on="image", validate="one_to_one")
per_image["delta_dsc"] = per_image["dsc_optimized"] - per_image["dsc_baseline"]

mask_dirs = {v: PIPELINE_DIR / "phase5_test" / "masks" / f"{v}_{MODEL}" for v in VARIANTS}
missing = [str(d) for d in mask_dirs.values() if not d.is_dir()]
if missing:
    raise FileNotFoundError(f"Predicted masks not found: {missing}. Re-run Phase 5a without --no-save-masks.")

summary = pd.DataFrame({
    v: {"mean DSC": per_image[f"dsc_{v}"].mean(), "mean JSI": per_image[f"jsi_{v}"].mean(),
        "empty predictions": int(per_image[f"empty_pred_{v}"].sum())}
    for v in VARIANTS
})
print(f"model={MODEL}  test images={len(per_image)}  "
      f"improved={(per_image.delta_dsc > 0).sum()}  worse={(per_image.delta_dsc < 0).sum()}")
summary.round(4)

In [ ]:
def load_image(image: str) -> np.ndarray:
    img = cv2.imread(str(local_path(image)))
    if img is None:
        raise FileNotFoundError(local_path(image))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def load_pred(variant: str, image: str) -> np.ndarray:
    m = cv2.imread(str(mask_dirs[variant] / f"{Path(image).stem}.png"), cv2.IMREAD_GRAYSCALE)
    return m > 127


def load_gt(image: str, h: int, w: int) -> np.ndarray:
    return rasterize_yolo_label(label_path_for(local_path(image)), h, w)


def overlay(ax, img: np.ndarray, gt: np.ndarray, pred: np.ndarray, title: str) -> None:
    # Overlays are drawn on a lightened greyscale copy so the fill colours read true on dark
    # lesions (yellow over dark brown would turn olive); the colour original is in column 1.
    grey = img.astype(float).mean(axis=2, keepdims=True) / 255
    out = np.repeat(0.4 + 0.6 * grey, 3, axis=2)
    for region, color in ((gt & ~pred, GT_COLOR), (pred & ~gt, PRED_COLOR), (gt & pred, OVERLAP_COLOR)):
        out[region] = (1 - ALPHA) * out[region] + ALPHA * np.array(to_rgb(color))
    ax.imshow(out)
    if gt.any():
        ax.contour(gt.astype(float), levels=[0.5], colors=[GT_COLOR], linewidths=1.2, linestyles="solid")
    if pred.any():
        ax.contour(pred.astype(float), levels=[0.5], colors=[PRED_COLOR], linewidths=1.2, linestyles="dashed")
    ax.set_title(title, fontsize=8)
    ax.axis("off")


LEGEND = [
    Patch(facecolor=GT_COLOR, alpha=ALPHA, edgecolor=GT_COLOR, label="Ground truth only (FN) — solid border"),
    Patch(facecolor=PRED_COLOR, alpha=ALPHA, edgecolor=PRED_COLOR, label="Prediction only (FP) — dashed border"),
    Patch(facecolor=OVERLAP_COLOR, alpha=ALPHA, edgecolor=OVERLAP_COLOR, label="Overlap (TP)"),
]


def show_rows(rows: pd.DataFrame, heading: str, fname: str) -> None:
    """One row per image: image | Baseline overlay | Optimized overlay."""
    fig, axes = plt.subplots(len(rows), 3, figsize=(7.2, 2.5 * len(rows)), squeeze=False)
    for ax_row, (_, r) in zip(axes, rows.iterrows()):
        img = load_image(r.image)
        gt = load_gt(r.image, *img.shape[:2])
        ax_row[0].imshow(img)
        ax_row[0].set_title(Path(r.image).stem[:28], fontsize=8)
        ax_row[0].axis("off")
        for ax, v in zip(ax_row[1:], VARIANTS):
            overlay(ax, img, gt, load_pred(v, r.image),
                    f"{v.capitalize()} · DSC {r[f'dsc_{v}']:.3f} · JSI {r[f'jsi_{v}']:.3f}")
    fig.suptitle(f"{heading} — YOLO26-{MODEL}-seg, test set", x=0.01, ha="left", fontweight="semibold")
    fig.legend(handles=LEGEND, loc="lower center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.03, 1, 0.98))
    save(fig, fname)
    plt.show()

## Random test images
A seeded random sample (`SEED`), so the same images are shown on every run.

In [ ]:
sample = per_image.sample(n=min(N_SAMPLES, len(per_image)), random_state=SEED)
show_rows(sample, "Random sample", f"seg_random_{MODEL}")

## Where HPO helped most
Largest per-image DSC gains (Optimized − Baseline).

In [ ]:
show_rows(per_image.nlargest(min(N_EXTREMES, len(per_image)), "delta_dsc"), "Largest HPO gains", f"seg_gains_{MODEL}")

## Where HPO hurt most
Largest per-image DSC regressions — useful to check whether the optimized model trades robustness for average accuracy.

In [ ]:
show_rows(per_image.nsmallest(min(N_EXTREMES, len(per_image)), "delta_dsc"), "Largest HPO regressions", f"seg_regressions_{MODEL}")

## Hardest images for the optimized model
Lowest optimized DSC — failure analysis (low contrast, hair, artefacts, very large/small lesions).

In [ ]:
show_rows(per_image.nsmallest(min(N_EXTREMES, len(per_image)), "dsc_optimized"), "Hardest cases (optimized)", f"seg_hardest_{MODEL}")